# Preprocessing Pipeline


## Imports

In [ ]:
import os
import numpy as np
import random
import requests
import matplotlib.pyplot as plt
import zipfile
import gdown
from tqdm import tqdm


work_dir = os.getcwd()
print("work_dir:", work_dir)


work_dir: /content/battery


# Download Raw Data

In [2]:
RAW_DIR = os.path.join(work_dir, "data_raw")
os.makedirs(RAW_DIR, exist_ok=True)

FILE_ID = "1Nh7nIxr3Gu393cahzu1ta_eNsVLnyDEm"
ZIP_PATH = os.path.join(RAW_DIR, "batteries_preprocessed_dataset.zip")

gdown.download(id=FILE_ID, output=ZIP_PATH, quiet=False)

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(RAW_DIR)
os.remove(ZIP_PATH)

num_files = len([f for f in os.listdir(RAW_DIR) if f.endswith(".npy")])
print(f"Total .npy files extracted: {num_files}")


Downloading...
From (original): https://drive.google.com/uc?id=1Nh7nIxr3Gu393cahzu1ta_eNsVLnyDEm
From (redirected): https://drive.google.com/uc?id=1Nh7nIxr3Gu393cahzu1ta_eNsVLnyDEm&confirm=t&uuid=21e49732-26d7-4df8-ad14-42875c3a148b
To: /content/battery/data_raw/batteries_preprocessed_dataset.zip
100%|██████████| 1.95G/1.95G [00:25<00:00, 75.8MB/s]


Total .npy files extracted: 228


## Quick Data Check

In [3]:
# Quick sanity check on one raw file
RAW_DIR = os.path.join(work_dir, "data_raw")
sample_raw = next(f for f in os.listdir(RAW_DIR) if f.endswith(".npy"))
x = np.load(os.path.join(RAW_DIR, sample_raw), allow_pickle=True)
print(f"File      : {sample_raw}")
print(f"Shape     : {x.shape}")
print(f"Dtype     : {x.dtype}")
print(f"First row : {x[0]}")


File      : processed_P022_3_S05_C04.npy
Shape     : (195902, 6)
Dtype     : float64
First row : [5.7000e+03 3.5526e+00 1.0000e-04 2.5172e+01 2.6560e+01 1.0000e+02]


# Reset Intermediate Directories
**Only run this section if you want to regenerate everything from scratch.**

In [4]:
d = os.path.join(work_dir, "data_processed")
try:
    for fn in os.listdir(d):
        fp = os.path.join(d, fn)
        if os.path.isfile(fp):
            os.remove(fp)
    os.rmdir(d)
    print(f"Removed: {d}")
except OSError as e:
    print(e)
d = os.path.join(work_dir, "data_downsampled")
try:
    for fn in os.listdir(d):
        fp = os.path.join(d, fn)
        if os.path.isfile(fp):
            os.remove(fp)
    os.rmdir(d)
    print(f"Removed: {d}")
except OSError as e:
    print(e)
d = os.path.join(work_dir, "data_windowed")
try:
    for fn in os.listdir(d):
        fp = os.path.join(d, fn)
        if os.path.isfile(fp):
            os.remove(fp)
    os.rmdir(d)
    print(f"Removed: {d}")
except OSError as e:
    print(e)
d = os.path.join(work_dir, "data_normalized")
try:
    for fn in os.listdir(d):
        fp = os.path.join(d, fn)
        if os.path.isfile(fp):
            os.remove(fp)
    os.rmdir(d)
    print(f"Removed: {d}")
except OSError as e:
    print(e)
d = os.path.join(work_dir, "data_group")
try:
    for fn in os.listdir(d):
        fp = os.path.join(d, fn)
        if os.path.isfile(fp):
            os.remove(fp)
    os.rmdir(d)
    print(f"Removed: {d}")
except OSError as e:
    print(e)
d = os.path.join(work_dir, "data_cnn")
try:
    for fn in os.listdir(d):
        fp = os.path.join(d, fn)
        if os.path.isfile(fp):
            os.remove(fp)
    os.rmdir(d)
    print(f"Removed: {d}")
except OSError as e:
    print(e)
d = os.path.join(work_dir, "data_cnn_normalized")
try:
    for fn in os.listdir(d):
        fp = os.path.join(d, fn)
        if os.path.isfile(fp):
            os.remove(fp)
    os.rmdir(d)
    print(f"Removed: {d}")
except OSError as e:
    print(e)


[Errno 2] No such file or directory: '/content/battery/data_processed'
[Errno 2] No such file or directory: '/content/battery/data_downsampled'
[Errno 2] No such file or directory: '/content/battery/data_windowed'
[Errno 2] No such file or directory: '/content/battery/data_normalized'
[Errno 2] No such file or directory: '/content/battery/data_group'
[Errno 2] No such file or directory: '/content/battery/data_cnn'
[Errno 2] No such file or directory: '/content/battery/data_cnn_normalized'


# Downsampling


In [5]:
# Config
DOWNSAMPLE_MODE = "time"  # None | "factor" | "time"
DOWNSAMPLE_FACTOR = 5  # used when mode == "factor"
SOURCE_INTERVAL_SEC = 30  # original sampling interval (seconds)
TARGET_INTERVAL_SEC = 1800  # target interval (1800 = 30 minutes)
TIME_COL = 0


def downsample_by_time(x, src_sec, tgt_sec, time_col=0):
    t = x[:, time_col]
    t_rel = t - t[0]
    bin_idx = (t_rel / tgt_sec).astype(int)
    _, last = np.unique(bin_idx[::-1], return_index=True)
    last = len(bin_idx) - 1 - last
    last.sort()
    return x[last]


def downsample_file(x):
    if DOWNSAMPLE_MODE is None:
        return x
    if DOWNSAMPLE_MODE == "factor":
        return x[::DOWNSAMPLE_FACTOR]
    if DOWNSAMPLE_MODE == "time":
        return downsample_by_time(x, SOURCE_INTERVAL_SEC, TARGET_INTERVAL_SEC, TIME_COL)
    raise ValueError(f"Unknown DOWNSAMPLE_MODE: {DOWNSAMPLE_MODE!r}")


# Run
ds_in = os.path.join(work_dir, "data_raw")
ds_out = os.path.join(work_dir, "data_downsampled")
os.makedirs(ds_out, exist_ok=True)

for f in tqdm(os.listdir(ds_in)):
    if not f.endswith(".npy"):
        continue
    x = np.load(os.path.join(ds_in, f), allow_pickle=True)
    np.save(os.path.join(ds_out, f), downsample_file(x))

sample = [f for f in os.listdir(ds_out) if f.endswith(".npy")][0]
_b = np.load(os.path.join(ds_in, sample), allow_pickle=True)
_a = np.load(os.path.join(ds_out, sample))
print(f"Mode   : {DOWNSAMPLE_MODE}")
print(f"Before : {_b.shape}  →  After : {_a.shape}")
print(f"Files  : {len([f for f in os.listdir(ds_out) if f.endswith('.npy')])}")


100%|██████████| 228/228 [00:13<00:00, 16.99it/s]

Mode   : time
Before : (195902, 6)  →  After : (3266, 6)
Files  : 228


# Drop SoC Column
Raw: [time, voltage, current, temperature, **SoC**, SoH]  
Processed: [time, voltage, current, temperature, SoH]

In [6]:
proc_in = os.path.join(work_dir, "data_downsampled")
proc_out = os.path.join(work_dir, "data_processed")
os.makedirs(proc_out, exist_ok=True)

for f in os.listdir(proc_in):
    if not f.endswith(".npy"):
        continue
    x = np.load(os.path.join(proc_in, f), allow_pickle=True)
    np.save(os.path.join(proc_out, f), np.delete(x, 4, axis=1))

sample = [f for f in os.listdir(proc_out) if f.endswith(".npy")][0]
xs = np.load(os.path.join(proc_out, sample))
print(f"Processed shape (should be N×5): {xs.shape}")
print(f"Columns: [time, voltage, current, temperature, SoH]")


Processed shape (should be N×5): (3266, 5)
Columns: [time, voltage, current, temperature, SoH]


# Windowing GRU / LSTM / XGBoost Dataset

**Features per window: 5** [voltage, current, temperature, dV/dt, dI/dt]  
**Label:** SoH at the *last* timestep of the window 


## Windowing Function

In [ ]:
# Windowing config
STRIDE_MODE = "fixed"
TARGET_WINS = 1  # used only in dynamic mode
WINDOW_SIZE = 2000
FIXED_STRIDE = 250


def create_windows(
    x, window_size, target_windows, stride_mode="fixed", fixed_stride=145
):
    n = x.shape[0]
    stride = (
        fixed_stride
        if stride_mode == "fixed"
        else max(1, (n - window_size) // target_windows)
    )

    X, y = [], []
    for i in range(0, n - window_size + 1, stride):
        window = x[i : i + window_size]
        features = window[:, 1:-1].copy()

        # Derivative features
        dv = np.gradient(features[:, 0])
        di = np.gradient(features[:, 1])
        features = np.concatenate([features, dv[:, None], di[:, None]], axis=1)

        X.append(features)
        y.append(window[-1, -1])  # last SoH in window

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


## Main Loop

In [ ]:
wind_in = os.path.join(work_dir, "data_processed")
wind_out = os.path.join(work_dir, "data_windowed")
os.makedirs(wind_out, exist_ok=True)

skipped = 0
for f in tqdm(os.listdir(wind_in)):
    if not f.endswith(".npy"):
        continue
    x = np.load(os.path.join(wind_in, f))
    if x.shape[0] < WINDOW_SIZE:
        skipped += 1
        continue
    X, y = create_windows(
        x, WINDOW_SIZE, TARGET_WINS, stride_mode=STRIDE_MODE, fixed_stride=FIXED_STRIDE
    )
    base = f.replace(".npy", "")
    np.save(os.path.join(wind_out, f"X_{base}.npy"), X)
    np.save(os.path.join(wind_out, f"y_{base}.npy"), y)

files = [f for f in os.listdir(wind_out) if f.startswith("X_")]
print(f"Windowed files : {len(files)}  |  Skipped (too short): {skipped}")
sample_X = np.load(os.path.join(wind_out, files[0]))
print(f"Window shape   : {sample_X.shape}  → (N_windows, {WINDOW_SIZE}, 5_features)")
print(f"Features       : [voltage, current, temperature, dV/dt, dI/dt]")


# Battery-Aware Train / Val / Test Split

## Config

In [ ]:
SEED = 45
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
random.seed(SEED)


def get_battery_id(fname):
    parts = fname.replace("X_", "").replace("y_", "").split("_")
    for p in parts:
        if p.startswith("P") and p[1:].isdigit():
            return p
    return parts[1]


## Split & Load

In [ ]:
wind_dir = os.path.join(work_dir, "data_windowed")
norm_dir = os.path.join(work_dir, "data_normalized")
os.makedirs(norm_dir, exist_ok=True)

# Battery-aware split
x_files = [f for f in os.listdir(wind_dir) if f.startswith("X_")]
bat_dict = {}
for f in x_files:
    bat_dict.setdefault(get_battery_id(f), []).append(f)

batteries = list(bat_dict.keys())
random.shuffle(batteries)
n = len(batteries)
train_b = batteries[: int(TRAIN_RATIO * n)]
val_b = batteries[int(TRAIN_RATIO * n) : int((TRAIN_RATIO + VAL_RATIO) * n)]
test_b = batteries[int((TRAIN_RATIO + VAL_RATIO) * n) :]

print(f"Batteries → train:{len(train_b)}  val:{len(val_b)}  test:{len(test_b)}")


def load_split(blist):
    Xs, ys = [], []
    for b in blist:
        for f in bat_dict[b]:
            Xs.append(np.load(os.path.join(wind_dir, f)))
            ys.append(np.load(os.path.join(wind_dir, f.replace("X_", "y_"))))
    return np.concatenate(Xs), np.concatenate(ys)


X_train, y_train = load_split(train_b)
X_val, y_val = load_split(val_b)
X_test, y_test = load_split(test_b)

print(f"Train : {X_train.shape}  y=[{y_train.min():.2f},{y_train.max():.2f}]")
print(f"Val   : {X_val.shape}    y=[{y_val.min():.2f},{y_val.max():.2f}]")
print(f"Test  : {X_test.shape}   y=[{y_test.min():.2f},{y_test.max():.2f}]")


## Label Distribution

In [ ]:
plt.figure(figsize=(8, 3))
plt.hist(y_train, bins=60, alpha=0.6, label="train")
plt.hist(y_val, bins=60, alpha=0.6, label="val")
plt.hist(y_test, bins=60, alpha=0.6, label="test")
plt.title("SoH label distribution per split")
plt.xlabel("SoH")
plt.ylabel("Count")
plt.legend()
plt.tight_layout()
plt.show()


## Scaling

In [ ]:
SCALING_MODE = "standardize"  # "standardize" | "normalize" | None


def scale_fit(X_train):
    if SCALING_MODE == "standardize":
        p1 = X_train.mean(axis=(0, 1), keepdims=True)
        p2 = X_train.std(axis=(0, 1), keepdims=True) + 1e-8
    elif SCALING_MODE == "normalize":
        p1 = X_train.min(axis=(0, 1), keepdims=True)
        p2 = (X_train.max(axis=(0, 1), keepdims=True) - p1) + 1e-8
    else:
        raise ValueError(f"Unknown SCALING_MODE: {SCALING_MODE!r}")
    return p1, p2


def scale_apply(X, p1, p2):
    return (X - p1) / p2


p1, p2 = scale_fit(X_train)
X_train = scale_apply(X_train, p1, p2)
X_val = scale_apply(X_val, p1, p2)
X_test = scale_apply(X_test, p1, p2)

print(f"Scaling mode     : {SCALING_MODE}")
print(f"Train mean (post): {X_train.mean(axis=(0, 1)).round(4)}")
print(f"Train std  (post): {X_train.std(axis=(0, 1)).round(4)}")


## Save GRU/XGB Dataset

In [ ]:
np.save(os.path.join(norm_dir, "X_train.npy"), X_train)
np.save(os.path.join(norm_dir, "y_train.npy"), y_train)
np.save(os.path.join(norm_dir, "X_val.npy"), X_val)
np.save(os.path.join(norm_dir, "y_val.npy"), y_val)
np.save(os.path.join(norm_dir, "X_test.npy"), X_test)
np.save(os.path.join(norm_dir, "y_test.npy"), y_test)
# Save scale params for inference-time normalisation
np.save(os.path.join(norm_dir, "scale_p1.npy"), p1)
np.save(os.path.join(norm_dir, "scale_p2.npy"), p2)
print("GRU/XGB dataset saved to:", norm_dir)
print(f"  X shape: {X_train.shape}   Features: [V, I, T, dV/dt, dI/dt]")


# Group Labels for XGBoost CV

In [ ]:
group_dir = os.path.join(work_dir, "data_group")
os.makedirs(group_dir, exist_ok=True)

groups = []
for f in os.listdir(wind_dir):
    if not f.startswith("X_"):
        continue
    bid = get_battery_id(f)
    if bid not in train_b:
        continue
    y = np.load(os.path.join(wind_dir, f.replace("X_", "y_")))
    groups.extend([bid] * len(y))

groups = np.array(groups)
np.save(os.path.join(group_dir, "groups_train.npy"), groups)
print(f"Groups shape : {groups.shape}")
print(f"Unique bats  : {len(np.unique(groups))}")


# CNN Dataset Preprocessing

## CNN Config & Windowing Function

In [ ]:
# CNN windowing config
CNN_WINDOW_SIZE = 1166
CNN_STRIDE_MODE = "fixed"  # "fixed" | "dynamic"
CNN_FIXED_STRIDE = 145
CNN_TARGET_WINS = 16  # used in dynamic mode


def compute_derivatives(data):
    dv = np.gradient(data[:, 1])  # dV/dt
    di = np.gradient(data[:, 2])  # dI/dt
    return dv, di


def create_cnn_windows(data, stride_mode="dynamic", fixed_stride=None):
    dv, di = compute_derivatives(data)
    features = np.stack(
        [
            data[:, 1],  # voltage
            data[:, 2],  # current
            data[:, 3],  # temperature
            dv,
            di,
        ],
        axis=1,
    )
    target = data[:, -1]

    if stride_mode == "fixed":
        stride = fixed_stride if fixed_stride is not None else CNN_WINDOW_SIZE
    elif stride_mode == "dynamic":
        stride = max(1, (len(data) - CNN_WINDOW_SIZE) // CNN_TARGET_WINS)
    else:
        raise ValueError(f"Unknown stride_mode: {stride_mode!r}")

    X, y = [], []
    for i in range(0, len(data) - CNN_WINDOW_SIZE, stride):
        X.append(features[i : i + CNN_WINDOW_SIZE])
        y.append(target[i + CNN_WINDOW_SIZE - 1])  # last SoH in window
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


## CNN Main Loop

In [ ]:
cnn_in = os.path.join(work_dir, "data_processed")
cnn_out = os.path.join(work_dir, "data_cnn")
os.makedirs(cnn_out, exist_ok=True)

skipped_cnn = 0
for f in tqdm([fn for fn in os.listdir(cnn_in) if fn.endswith(".npy")]):
    data = np.load(os.path.join(cnn_in, f))
    if data.shape[0] < CNN_WINDOW_SIZE:
        skipped_cnn += 1
        continue
    X, y = create_cnn_windows(
        data, stride_mode=CNN_STRIDE_MODE, fixed_stride=CNN_FIXED_STRIDE
    )
    if len(X) == 0:
        skipped_cnn += 1
        continue
    np.save(os.path.join(cnn_out, f"X_{f}"), X)
    np.save(os.path.join(cnn_out, f"y_{f}"), y)

cfiles = [f for f in os.listdir(cnn_out) if f.startswith("X_")]
print(f"CNN windowed files : {len(cfiles)}  |  Skipped: {skipped_cnn}")
if cfiles:
    sx = np.load(os.path.join(cnn_out, cfiles[0]))
    print(f"CNN window shape   : {sx.shape}  → (N, {CNN_WINDOW_SIZE}, 5_channels)")
    print(f"CNN channels       : [voltage, current, temperature, dV/dt, dI/dt]")


## CNN Split & Load

In [ ]:
random.seed(SEED)
cnn_data_dir = os.path.join(work_dir, "data_cnn")
cnn_norm_dir = os.path.join(work_dir, "data_cnn_normalized")
os.makedirs(cnn_norm_dir, exist_ok=True)

cnn_x_files = [f for f in os.listdir(cnn_data_dir) if f.startswith("X_")]
cnn_bat_dict = {}
for f in cnn_x_files:
    cnn_bat_dict.setdefault(get_battery_id(f), []).append(f)

cnn_bats = list(cnn_bat_dict.keys())
random.shuffle(cnn_bats)
n = len(cnn_bats)
cnn_train_b = cnn_bats[: int(TRAIN_RATIO * n)]
cnn_val_b = cnn_bats[int(TRAIN_RATIO * n) : int((TRAIN_RATIO + VAL_RATIO) * n)]
cnn_test_b = cnn_bats[int((TRAIN_RATIO + VAL_RATIO) * n) :]


def cnn_load_split(blist):
    Xs, ys = [], []
    for b in blist:
        for f in cnn_bat_dict[b]:
            Xs.append(np.load(os.path.join(cnn_data_dir, f)))
            ys.append(np.load(os.path.join(cnn_data_dir, f.replace("X_", "y_"))))
    return np.concatenate(Xs), np.concatenate(ys)


Xc_train, yc_train = cnn_load_split(cnn_train_b)
Xc_val, yc_val = cnn_load_split(cnn_val_b)
Xc_test, yc_test = cnn_load_split(cnn_test_b)

print(f"CNN Train : {Xc_train.shape}  y=[{yc_train.min():.2f},{yc_train.max():.2f}]")
print(f"CNN Val   : {Xc_val.shape}    y=[{yc_val.min():.2f},{yc_val.max():.2f}]")
print(f"CNN Test  : {Xc_test.shape}   y=[{yc_test.min():.2f},{yc_test.max():.2f}]")


## CNN Scaling

In [ ]:
cnn_p1, cnn_p2 = scale_fit(Xc_train)
Xc_train = scale_apply(Xc_train, cnn_p1, cnn_p2)
Xc_val = scale_apply(Xc_val, cnn_p1, cnn_p2)
Xc_test = scale_apply(Xc_test, cnn_p1, cnn_p2)

print(f"CNN train mean (post): {Xc_train.mean(axis=(0, 1)).round(4)}")
print(f"CNN train std  (post): {Xc_train.std(axis=(0, 1)).round(4)}")


## Save CNN Dataset

In [ ]:
np.save(os.path.join(cnn_norm_dir, "X_train.npy"), Xc_train)
np.save(os.path.join(cnn_norm_dir, "y_train.npy"), yc_train)
np.save(os.path.join(cnn_norm_dir, "X_val.npy"), Xc_val)
np.save(os.path.join(cnn_norm_dir, "y_val.npy"), yc_val)
np.save(os.path.join(cnn_norm_dir, "X_test.npy"), Xc_test)
np.save(os.path.join(cnn_norm_dir, "y_test.npy"), yc_test)
np.save(os.path.join(cnn_norm_dir, "scale_p1.npy"), cnn_p1)
np.save(os.path.join(cnn_norm_dir, "scale_p2.npy"), cnn_p2)
print("CNN dataset saved to:", cnn_norm_dir)
print(f"  X shape: {Xc_train.shape}   Channels: [V, I, T, dV/dt, dI/dt]")


# Final Summary Check

In [ ]:
print("=" * 55)
print("FINAL DATASET SUMMARY")
print("=" * 55)
print(f"GRU/XGB  train: {X_train.shape}   features: 5 (V,I,T,dV,dI)")
print(f"GRU/XGB  val  : {X_val.shape}")
print(f"GRU/XGB  test : {X_test.shape}")
print(f"y_train range : [{y_train.min():.3f}, {y_train.max():.3f}]")
print()
print(f"CNN      train: {Xc_train.shape}   channels: 5 (V,I,T,dV,dI)")
print(f"CNN      val  : {Xc_val.shape}")
print(f"CNN      test : {Xc_test.shape}")
print(f"yc_train range: [{yc_train.min():.3f}, {yc_train.max():.3f}]")
print()
print(f"Groups   train: {groups.shape}  unique batteries: {len(np.unique(groups))}")
print("=" * 55)
print("Both datasets use label = last SoH in window (not mean).")
print("Both datasets include dV/dt and dI/dt as features.")
